In [267]:
from pathlib import Path
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from sklearn.model_selection import train_test_split
from peft import LoraConfig, TaskType
from torch.utils.data import Dataset
from peft import LoraConfig, get_peft_model, TaskType
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

random_state = 2026

In [268]:
def generate_dataset(path="/content/diabetes_binary_health_indicators_BRFSS2015.csv"):
  df = pd.read_csv(path)
  rename_cols = {
      "Diabetes_binary": "diabetes",
      "MentHlth": "days_mentally_ill",
      "Smoker": "smoking",
      "HvyAlcoholConsump": "alchohol_consumption",
      "PhysActivity": "physical_activity",
      "Fruits": "consumed_fruits",
      "Veggies": "consumed_vegetables",
      "PhysHlth": "days_physically_ill",
      "DiffWalk": "difficulty_walking",
      "GenHlth": "general_health",
  }
  columns_filter = [col for col in rename_cols.keys()]
  df = df[columns_filter]
  df = df.rename(columns=rename_cols)
  df["diabetes"] = df.diabetes.astype(int)

  df = df.dropna()
  return df

def undersample_dataset(dataset):

  df_diabetic = dataset[dataset.diabetes == 1]
  df_non_diabetic = dataset[dataset.diabetes == 0]

  df_downsampled = df_non_diabetic.sample(n=len(df_diabetic), random_state=random_state)
  df_balanced = pd.concat([df_downsampled, df_diabetic])
  df_balanced = df_balanced.reset_index(drop=True)

  return df_balanced

def text_format(train_dataset):
  texts = []
  for i in range(len(train_dataset)):
    features = train_dataset.iloc[i]

    consumed_vegetables = "Yes"  if features.consumed_vegetables == 1 else "No"
    consumed_fruits = "Yes"  if features.consumed_fruits == 1 else "No"
    smoking = "Yes" if features.smoking == 1 else "No"
    alcohol = "Yes" if features.alchohol_consumption == 1 else "No"
    difficulty_walking = "Yes" if features.difficulty_walking == 1 else "No"
    physically_active = "Yes" if features.physical_activity == 1 else "No"


    build_up = []
    build_up.append(f"Eats 1 or more vegetable a day: {consumed_vegetables};")
    build_up.append(f"Eats 1 or more fruit a day: {consumed_fruits};")
    build_up.append(f"Number of days spent mentally ill for last 30 days: {features.days_mentally_ill};")
    build_up.append(f"Number of days spent physically ill for last 30 days: {features.days_physically_ill};")
    build_up.append(f"Has been physically active in the last 30 days: {physically_active};")
    build_up.append(f"Has difficulty walking stairs: {difficulty_walking};")
    build_up.append(f"Overall health score from self assessment (1=excellent 2=very good 3=good 4=fair 5=poor): {features.general_health};")
    build_up.append(f"Has smoked more than 100 cigarettes: {smoking};")
    build_up.append(f"Heavy alchohol consumption (men=14 or more drinks a week, women=7 or more drink a week): {alcohol};")

    text = "".join(build_up)

    texts.append(text)
  return texts


In [281]:
df = generate_dataset()
num_diabetes = len(df[df.diabetes == 1])
num_no_diabetes = len(df[df.diabetes == 0])
print(f"Before undersampling - number of diabetic participants: {num_diabetes}, number of non-diabetic participants: {num_no_diabetes}")

df = undersample_dataset(df)

num_diabetes = len(df[df.diabetes == 1])
num_no_diabetes = len(df[df.diabetes == 0])

print(f"After undersampling - number of diabetic participants: {num_diabetes}, number of non-diabetic participants: {num_no_diabetes}")

Before undersampling - number of diabetic participants: 35346, number of non-diabetic participants: 218334
After undersampling - number of diabetic participants: 35346, number of non-diabetic participants: 35346


In [297]:
df

,diabetes,days_mentally_ill,smoking,alchohol_consumption,physical_activity,consumed_fruits,consumed_vegetables,days_physically_ill,difficulty_walking,general_health
0,0,20.0,1.0,0.0,1.0,0.0,1.0,30.0,0.0,5.0
1,0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,3.0
2,0,30.0,1.0,0.0,0.0,0.0,1.0,7.0,1.0,3.0
3,0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,2.0
4,0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...
70687,1,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,4.0
70688,1,0.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,2.0
70689,1,15.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,5.0
70690,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4.0


In [328]:
tokenizer = AutoTokenizer.from_pretrained("dmis-lab/biobert-base-cased-v1.1")
model = AutoModelForSequenceClassification.from_pretrained("dmis-lab/biobert-base-cased-v1.1", num_labels=2, dtype=torch.bfloat16)
tokenizer.pad_token = '[PAD]'

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: dmis-lab/biobert-base-cased-v1.1
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were ne

In [325]:
temp_df = df

y = temp_df["diabetes"]
X = temp_df.drop(columns=["diabetes"])

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7, random_state=random_state, stratify=y, shuffle=True)

,diabetes
26951,0
8217,0
52637,1
50482,1
63279,1
...,...
702,0
52682,1
70437,1
50975,1


In [307]:
class DiabetesDataset(Dataset):

  def __init__(self, encodings, labels):
    self.encodings = encodings
    self.labels = labels

  def __getitem__(self, idx):
    result = {key: val[idx] for key, val in self.encodings.items()}
    result["labels"] = self.labels[idx]
    return result

  def __len__(self):
    return len(self.labels)

In [308]:
texts = text_format(X_train)
train_encodings = tokenizer(texts, truncation=True, padding=True, return_tensors="pt")
texts = text_format(X_test)
test_encodings = tokenizer(texts, truncation=True, padding=True, return_tensors="pt")



In [326]:
def compute_classification_metrics(pred):
    logits, labels = pred
    preds = logits.argmax(axis=-1)
    f1 = f1_score(labels, preds)
    precision = precision_score(labels, preds)
    accuracy = accuracy_score(labels, preds)
    recall = recall_score(labels, preds)
    return {
        "precision": precision,
        "recall": recall,
        "accuracy": accuracy,
        "f1": f1,
    }

In [327]:
peft_config = LoraConfig(task_type=TaskType.SEQ_CLS, inference_mode=False, r=8, lora_alpha=32, lora_dropout=0.1)
model = get_peft_model(model, peft_config)

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [312]:
model.print_trainable_parameters()

trainable params: 296,450 || all params: 108,608,260 || trainable%: 0.2730


In [313]:
training_args = TrainingArguments(
    output_dir="./train_args",
    learning_rate=1e-3,
    per_device_train_batch_size=128,
    per_device_eval_batch_size=128,
    num_train_epochs=1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=DiabetesDataset(train_encodings, y_train.values),
    eval_dataset=DiabetesDataset(test_encodings, y_test.values),
    compute_metrics=compute_classification_metrics
)




In [314]:
trainer.train()


Epoch,Training Loss,Validation Loss,Precision,Recall,Accuracy,F1
1,No log,0.693670,0.500000,1.000000,0.500000,0.666667
2,0.700945,0.693591,0.000000,0.000000,0.500000,0.000000


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


TrainOutput(global_step=774, training_loss=0.6991531411498708, metrics={'train_runtime': 47.4167, 'train_samples_per_second': 2087.195, 'train_steps_per_second': 16.323, 'total_flos': 7042771924168320.0, 'train_loss': 0.6991531411498708, 'epoch': 2.0})

In [315]:
eval_results = trainer.evaluate()
print(eval_results)

{'eval_loss': 0.6935914754867554, 'eval_precision': 0.0, 'eval_recall': 0.0, 'eval_accuracy': 0.5, 'eval_f1': 0.0, 'eval_runtime': 3.4281, 'eval_samples_per_second': 6186.51, 'eval_steps_per_second': 48.423, 'epoch': 2.0}


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [322]:
pred_results = trainer.predict(DiabetesDataset(test_encodings, y_test.values))

# Get predicted class labels
preds = pred_results.predictions.argmax(axis=-1)


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [321]:


# Count the number of 0s and 1s
unique, counts = np.unique(preds, return_counts=True)
print(dict(zip(unique, counts)))

{np.int64(0): np.int64(21208)}
